# Ilera-ATLAS · 03_translate: N-ATLaS translates reviewed items into Hausa and Yoruba

**Runbook Part 5.** Run it **twice**:

| Run | `STAGE` | When | What | Review |
|---|---|---|---|---|
| 1 | `"bench"` | **Sun 4 Oct** | 150 Hausa + 150 Yoruba benchmark items | native speaker checks **every** row before the Mon 5 Oct freeze |
| 2 | `"train"` | Mon 5 Oct (Commit mode) | 1,000 Hausa + 1,000 Yoruba training items | all flagged rows + 1 in 5 of the rest, by Tue 6 Oct |

**Input:** the English sheet after Part 4 review. In Google Sheets: File → Download → CSV for each tab,
combine them into **`review_en_final.csv`** (same columns as the drafts), then upload it to your Hugging
Face dataset under `reviewed/` (dataset page → Files → Add file → Upload files).

**How translation works (safer than translating whole answers):** the notebook translates the
*question*, the *WHAT TO DO steps* and the *DANGER SIGNS* separately, then rebuilds the answer itself. So
`ACTION:` (TREAT / REFER / URGENT), the four labels and the `SOURCE:` line stay in exact English and the
same parser works in every language.

**Automatic flags** tell reviewers where to look first: changed numbers/doses, wrong language
(GlotLID), suspicious length, repetition loops, untranslated text.

**Kaggle:** GPU T4 x2 · Internet On · secret `HF_TOKEN`. Smoke test first with `LIMIT = 6`.

In [ ]:
# ---- Config -------------------------------------------------------------
HF_USER = "YOUR_HF_USERNAME"          # <- change this
DATA_REPO = f"{HF_USER}/ilera-data"
BASE_MODEL = "NCAIR1/N-ATLaS"
STAGE = "bench"                        # "bench" (Sun) or "train" (Mon)
N_PER_LANG = {"bench": 150, "train": 1000}
LANGS = ["ha", "yo"]                   # add "ig" if you have an Igbo reviewer
LIMIT = 6                              # smoke test: items per language. Set None for the full run.
BATCH = 8                              # strings per generation batch (short texts); lower if OOM
MAX_NEW = 600
SAVE_EVERY = 25                        # batches between progress uploads
UPLOAD = True
EN_CSV = "reviewed/review_en_final.csv"   # path inside the HF dataset
LOCAL_EN_CSV = None                    # or a local path (offline testing / Kaggle input file)
USE_LID = True                         # GlotLID language check (downloads a 1.6 GB model once)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth", "fasttext-wheel"], check=True)
# If the next cells fail with an import error: Run > Restart kernel, then Run All again.

In [ ]:
import os
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_USER != "YOUR_HF_USERNAME", "Set HF_USER in the Config cell"
if UPLOAD or not LOCAL_EN_CSV:
    assert HF_TOKEN, "No HF_TOKEN found: add it as a Kaggle/Colab secret or env var"
    os.environ["HF_TOKEN"] = HF_TOKEN

## 1. Load the reviewed English items and pick what to translate
Only rows with `status` = `ok` or `fixed` are used. Benchmark rows must also have `key_facts`.
The Hausa and Yoruba benchmark sets are drawn from the **same** English benchmark items, so the three
languages form a parallel test: you can compare languages on identical questions.

In [ ]:
import csv, json, random, re, time, collections

path = LOCAL_EN_CSV
if not path:
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(DATA_REPO, EN_CSV, repo_type="dataset", token=HF_TOKEN)
rows = list(csv.DictReader(open(path, encoding="utf-8-sig")))
need = {"id", "split", "kind", "chunk_id", "source", "question", "action", "answer", "key_facts", "status"}
missing = need - set(rows[0])
assert not missing, f"review_en_final.csv is missing columns: {missing}"
rows = [r for r in rows if r["status"].strip().lower() in ("ok", "fixed")]
print(collections.Counter(r["split"] for r in rows), "reviewed English rows")

pool = [r for r in rows if r["split"] == STAGE]
if STAGE == "bench":
    no_facts = [r["id"] for r in pool if not r["key_facts"].strip()]
    assert not no_facts, f"{len(no_facts)} bench rows have no key_facts (e.g. {no_facts[:3]}): finish Part 4 first"
n = N_PER_LANG[STAGE]
assert len(pool) >= n, f"only {len(pool)} reviewed {STAGE} rows, need {n}"
picked = random.Random(1).sample(pool, n)          # same items for every language
if LIMIT:
    picked = picked[:LIMIT]
    print(f"*** SMOKE TEST: only {LIMIT} items per language. Set LIMIT = None for the full run. ***")
print(len(picked), "English items x", len(LANGS), "languages")

## 2. Split each answer into translatable parts
`ACTION` and `SOURCE` are kept from English. Only the steps and danger signs are translated.

In [ ]:
def split_answer(ans):
    action = re.search(r"ACTION:\s*(TREAT|REFER|URGENT)", ans).group(1)
    steps = re.search(r"WHAT TO DO:\s*\n?([\s\S]*?)(?=\n\s*DANGER SIGNS:|\n\s*SOURCE:|\Z)", ans)
    danger = re.search(r"DANGER SIGNS:\s*(.*)", ans)
    source = re.search(r"SOURCE:\s*(.*)", ans)
    return {"action": action,
            "steps": steps.group(1).strip() if steps else "",
            "danger": danger.group(1).strip() if danger else "none listed",
            "source": source.group(1).strip() if source else ""}

def build_answer(p):
    return (f"ACTION: {p['action']}\nWHAT TO DO:\n{p['steps']}\n"
            f"DANGER SIGNS: {p['danger']}\nSOURCE: {p['source']}")

_a = "ACTION: REFER\nWHAT TO DO:\n1. Give amoxicillin 250 mg.\n2. Refer today.\nDANGER SIGNS: convulsion\nSOURCE: CHEW Standing Orders 2024, Section 2.3 Fever, p. 50"
assert build_answer(split_answer(_a)) == _a
print("answer split/rebuild self-check passed")

## 3. Translation prompt
Rules given to N-ATLaS: keep numbers, doses, drug names and numbering; output the translation only.
Yoruba gets an extra instruction to write full tone marks (N-ATLaS's weakest language: 2.69/5 in its
own evaluation, so expect more fixes there).

In [ ]:
LANG_NAME = {"ha": "Hausa", "yo": "Yoruba", "ig": "Igbo"}
EXTRA = {"ha": "", "ig": "",
         "yo": "Write standard Yoruba with full tone marks and under-dots (for example: ọmọ, ìbà, ẹ̀jẹ̀). "}

T_PROMPT = """Translate the text below from English into {lang}.
Rules: keep all numbers, doses, units and drug names exactly as they are. Keep the numbered-list layout.
{extra}Write only the translation, nothing else.

Text:
{text}"""

def clean_output(s):
    s = s.strip()
    s = re.sub(r"^(Translation|Fassara|Ìtumọ̀|Itumo)\s*[:：]\s*", "", s, flags=re.I)
    return s.strip().strip('"').strip()

## 4. Load N-ATLaS (4-bit) and `chat()`

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tok = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=2048,
                                               load_in_4bit=True, token=HF_TOKEN)
FastLanguageModel.for_inference(model)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def chat(user_msgs, system=None, max_new=MAX_NEW):
    texts = []
    for u in user_msgs:
        msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": u}]
        texts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

print(chat([T_PROMPT.format(lang="Hausa", extra="", text="The child has fever for 3 days.")], max_new=60)[0])

## 5. Translate (resumable)
Each job is one string: `(English id, language, field)` with field = question, steps or danger.
Finished strings are stored in `translations_raw_{STAGE}.jsonl`; a re-run skips them.

In [ ]:
def upload(files, msg):
    if not UPLOAD:
        return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    for f in files:
        api.upload_file(path_or_fileobj=f, path_in_repo=f"translations/{f}", repo_id=DATA_REPO,
                        repo_type="dataset", commit_message=msg)

parts = {r["id"]: split_answer(r["answer"]) for r in picked}
jobs = []
for r in picked:
    for lang in LANGS:
        for field, text in (("question", r["question"]), ("steps", parts[r["id"]]["steps"]),
                            ("danger", parts[r["id"]]["danger"])):
            jobs.append((r["id"], lang, field, text))
jobs.sort(key=lambda j: len(j[3]))                     # similar lengths per batch = faster

RAW = f"translations_raw_{STAGE}.jsonl"
done = {}
if not os.path.exists(RAW) and UPLOAD:
    try:
        import shutil
        from huggingface_hub import hf_hub_download
        shutil.copy(hf_hub_download(DATA_REPO, f"translations/{RAW}", repo_type="dataset", token=HF_TOKEN), RAW)
        print("Resuming from Hugging Face copy of", RAW)
    except Exception:
        pass
if os.path.exists(RAW):
    for l in open(RAW, encoding="utf-8"):
        x = json.loads(l)
        done[(x["en_id"], x["lang"], x["field"])] = x["text"]
todo = [j for j in jobs if (j[0], j[1], j[2]) not in done]
print(f"{len(jobs)} strings | {len(done)} done | {len(todo)} to translate")

t0, nb = time.time(), (len(todo) + BATCH - 1) // BATCH
with open(RAW, "a", encoding="utf-8") as f:
    for b in range(nb):
        batch = todo[b * BATCH:(b + 1) * BATCH]
        outs = chat([T_PROMPT.format(lang=LANG_NAME[l], extra=EXTRA[l], text=t) for _, l, _, t in batch])
        for (eid, l, field, _), o in zip(batch, outs):
            done[(eid, l, field)] = clean_output(o)
            f.write(json.dumps({"en_id": eid, "lang": l, "field": field, "text": done[(eid, l, field)]},
                               ensure_ascii=False) + "\n")
        f.flush()
        el = time.time() - t0
        if (b + 1) % 5 == 0 or b + 1 == nb:
            print(f"batch {b + 1}/{nb}  {el / 60:.1f} min  ETA {el / (b + 1) * (nb - b - 1) / 60:.0f} min", flush=True)
        if (b + 1) % SAVE_EVERY == 0:
            upload([RAW], f"03_translate {STAGE}: progress {b + 1}/{nb}")
upload([RAW], f"03_translate {STAGE}: finished")

## 6. Automatic quality flags
| Flag | Meaning |
|---|---|
| `numbers` | a number/dose in the English text is missing from the translation |
| `lang` | GlotLID thinks the text is not Hausa/Yoruba |
| `length` | translation is much shorter or longer than the English |
| `repeat` | the model looped (same phrase repeated) |
| `copied` | output is (almost) the English text |
| `empty` | nothing came back |
| `no_tones` | Yoruba text with no tone marks at all |

Note: `fasttext.predict()` crashes with NumPy 2 (Kaggle's default), so the low-level `f.predict` is used.

In [ ]:
lid = None
if USE_LID:
    try:
        import fasttext
        from huggingface_hub import hf_hub_download
        lid = fasttext.load_model(hf_hub_download("cis-lmu/glotlid", "model.bin"))
    except Exception as e:
        print("GlotLID unavailable, skipping language check:", e)
LID_LABEL = {"ha": "__label__hau_Latn", "yo": "__label__yor_Latn", "ig": "__label__ibo_Latn"}
NUM = re.compile(r"\d+(?:[.,]\d+)?")
YO_TONES = re.compile(r"[àáèéìíòóùúẹọṣÀÁÈÉÌÍÒÓÙÚẸỌṢ\u0300\u0301]")

def flags_for(src, out, lang):
    f = []
    if not out.strip():
        return ["empty"]
    if set(NUM.findall(src)) - set(NUM.findall(out)):
        f.append("numbers")
    if len(src) > 25 and not (0.5 <= len(out) / len(src) <= 3.0):
        f.append("length")
    words = out.lower().split()
    grams = collections.Counter(tuple(words[i:i + 4]) for i in range(len(words) - 3))
    if grams and grams.most_common(1)[0][1] > 3:
        f.append("repeat")
    if src.strip().lower() == out.strip().lower() or (len(src) > 25 and src.lower()[:25] in out.lower()):
        f.append("copied")
    if lid is not None and len(out.split()) >= 4:
        label = lid.f.predict(out.replace("\n", " "), 1, 0.0, "strict")[0][1]
        if label != LID_LABEL[lang]:
            f.append("lang")
    if lang == "yo" and len(out.split()) >= 4 and not YO_TONES.search(out):
        f.append("no_tones")
    return f

assert flags_for("Give 250 mg twice daily", "Ba da mg sau biyu a rana", "ha") [:1] == ["numbers"]
assert "repeat" in flags_for("x " * 10, "a b c d a b c d a b c d a b c d a b c d", "ha")
print("flag self-check passed")

## 7. Build the review sheet
* `key_facts` are copied from English, but only facts that also appear in the translated answer are
  kept automatically (numbers and drug names usually survive translation; English words do not).
  `key_facts_en` keeps the original list so the reviewer can add the right Hausa/Yoruba words.
* `must_review` = yes for every benchmark row, and for training rows that are flagged or in a random
  1-in-5 sample.
* Flagged rows are sorted to the top.

In [ ]:
rng = random.Random(11)
out_rows = []
for r in picked:
    p = parts[r["id"]]
    for lang in LANGS:
        q = done.get((r["id"], lang, "question"), "")
        steps = done.get((r["id"], lang, "steps"), "")
        danger = done.get((r["id"], lang, "danger"), "")
        ans = build_answer({**p, "steps": steps, "danger": danger})
        fl = sorted(set(flags_for(r["question"], q, lang) + flags_for(p["steps"], steps, lang)
                        + ([] if p["danger"].lower() == "none listed" else flags_for(p["danger"], danger, lang))))
        facts_en = [k.strip() for k in r["key_facts"].split(";") if k.strip()]
        facts = [k for k in facts_en if k.lower() in ans.lower()]
        must = STAGE == "bench" or bool(fl) or rng.random() < 0.2
        out_rows.append({
            "id": r["id"].replace("EN-", lang.upper() + "-", 1), "en_id": r["id"], "lang": lang,
            "split": r["split"], "kind": r["kind"], "chunk_id": r["chunk_id"], "source": r["source"],
            "action": p["action"], "flags": ";".join(fl), "must_review": "yes" if must else "no",
            "question_en": r["question"], "question": q,
            "answer_en": r["answer"], "answer": ans,
            "key_facts_en": "; ".join(facts_en), "key_facts": "; ".join(facts),
            "status": "", "reviewer": "", "comment": ""})

out_rows.sort(key=lambda x: (x["lang"], x["flags"] == "", x["must_review"] == "no", x["id"]))
OUT = f"review_ha_yo_{STAGE}.csv"
with open(OUT, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(out_rows[0]))
    w.writeheader(); w.writerows(out_rows)
upload([OUT], f"03_translate {STAGE}: review sheet")

for lang in LANGS:
    L = [x for x in out_rows if x["lang"] == lang]
    fc = collections.Counter(fl for x in L for fl in x["flags"].split(";") if fl)
    print(f"{LANG_NAME[lang]}: {len(L)} rows | flagged {sum(bool(x['flags']) for x in L)} | "
          f"must_review {sum(x['must_review'] == 'yes' for x in L)} | flags {dict(fc)}")
ex = out_rows[0]
print("\nExample (", ex["id"], ex["flags"] or "no flags", ")\nEN:", ex["question_en"], "\n-> ", ex["question"], "\n" + ex["answer"])

## Reviewer instructions (paste into the sheet's first tab)
1. Filter `must_review = yes`. Flagged rows are on top.
2. Compare `question`/`answer` with the English columns. Fix the translation in place.
3. **Never change** `ACTION:`, the four labels or the `SOURCE:` line. Never change numbers or doses.
4. Yoruba: add correct tone marks and under-dots.
5. Benchmark rows: make `key_facts` 2–4 facts separated by `;` that appear **in the translated
   answer** (drug names and numbers are best; add the Hausa/Yoruba word where needed).
6. Set `status` = `ok`, `fixed` or `drop`, and your initials in `reviewer`.

When done, download as CSV and upload to Hugging Face as `reviewed/review_ha_yo_final.csv`
(bench and train rows together in one file; Part 6 reads it).

## Done when
* **Sun 4 Oct:** `translations/review_ha_yo_bench.csv` exists and is with the native speakers; reviewed
  version uploaded before the Mon 5 Oct freeze.
* **Mon 5 – Tue 6 Oct:** `translations/review_ha_yo_train.csv` reviewed (flagged + 1 in 5). If more than
  30% of the sampled rows of a language are bad, keep only that language's reviewed rows for training.